# MOMENT imputation — DIMER task-inference tutorial

**Profile:** `TASK-INFERENCE`  
**Notebook spec:** `1.0`  
**Capability:** pretrained reconstruction-backed time-series imputation  
**Upstream model:** `AutonLab/MOMENT-1-base` at immutable revision `9fea447e740eb968a9e8d80c7562ae122bdb5dde`

This notebook uses MOMENT's pretrained reconstruction path for patch-granular artificial masking. It withholds known source values, evaluates only deliberately hidden truth, preserves observed values in the exported imputed series, and writes machine-readable metrics and provenance. **No gradient training, fine-tuning, in-context conditioning, or fitted preprocessing occurs.**

**Upstream vs. this repository.** Upstream MOMENT supplies the pretrained reconstruction model. This repository supplies immutable pinning/integrity verification, long-format validation and canonicalization, patch-quantized masking semantics, masked-point evaluation, an observed-value-preserving imputed product, and provenance/export contracts.

**By the end of this notebook you will be able to:** verify runtime/model identity; load synthetic or BYOD data; validate/canonicalize input; hide one complete 8-step patch with known truth; impute and evaluate only withheld truth; compare a simple interpolation baseline; and export the imputed series, metrics, and provenance.

**This notebook does not demonstrate:** forecasting, classification, stochastic uncertainty intervals, or production fitness. The displayed MAE/RMSE values are sample/tutorial evidence only.

References: [repository README](https://github.com/kurtvalcorza/moment-pipeline), [model card](https://github.com/kurtvalcorza/moment-pipeline/blob/main/MODEL_CARD.md), [sample dataset card](https://github.com/kurtvalcorza/moment-pipeline/blob/main/examples/sample-data/DATASET_CARD.md), [upstream MOMENT](https://github.com/moment-timeseries-foundation-model/moment), and [pinned model repository](https://huggingface.co/AutonLab/MOMENT-1-base).


## Prerequisites and data contract

- **Runtime:** Python 3.12; CPU default; public v1 inference is `float32` only.
- **Network:** first run needs GitHub and Hugging Face access; pinned weights are ~454 MB. No credentials are required for the default public path.
- **Default data:** deterministic clean synthetic series generated by this repository. Metrics are tutorial evidence, not upstream benchmarks or deployment validation.
- **BYOD schema:** one UTF-8 CSV with `series_id`, `timestamp`, `channel`, `value`. `value` must be numeric or missing. Duplicate or ambiguous column names are rejected from the raw header before dataframe parsing, and duplicate `(series_id, channel, timestamp)` rows are rejected by production validation.
- **Operational ceilings:** 5,000,000 rows, 1,024 series, 32 channels, 1,024 windows; 512-step windows; 8-step non-overlapping patches. Long series keep the final 512 timestamps and short series are left-padded. Irregular spacing is surfaced rather than interpolated.
- **BYOD privacy:** the pipeline does not transmit the uploaded CSV to an external inference service. Colab uploads live in Google's hosted runtime; model files are fetched separately. Do not upload confidential, restricted, or sensitive data unless the runtime is authorized.

The default path is non-interactive. The artificial evaluation mask is deterministic, so no random seed is required. Exact floating-point outputs may still differ slightly across hardware/framework builds; no bitwise cross-platform reproducibility claim is made.


## 1. Bootstrap the repository and locked runtime

This stage installs the pinned `uv` bootstrap when needed, obtains the repository checkout, and installs the exported lock graph before installing this package without resolving a second dependency graph. Successful completion means the repository checkout is identified and the locked runtime is installed without intentional dependency drift; it does **not** yet verify or execute the model weights.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/kurtvalcorza/moment-pipeline.git"
REPO_NAME = "moment-pipeline"
UV_VERSION = "0.12.9"
ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", f"uv=={UV_VERSION}"],
        check=True,
    )
    if not Path(REPO_NAME).exists():
        subprocess.run(["git", "clone", "--depth", "1", "-q", REPO_URL], check=True)
    os.chdir(REPO_NAME)
    ROOT = Path.cwd()
    subprocess.run(
        ["uv", "pip", "install", "--system", "-r", "requirements.lock.txt"],
        check=True,
    )
    subprocess.run(
        ["uv", "pip", "install", "--system", "--no-deps", "-e", "."],
        check=True,
    )
else:
    print(f"Repository checkout detected: {ROOT}")
repo_commit = subprocess.run(
    ["git", "rev-parse", "HEAD"],
    cwd=ROOT,
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
print("repository commit:", repo_commit)


## 2. Inspect runtime, model identity, and limits

This stage exposes the effective Python/library versions, device and precision policy, immutable model identity, and operational ceilings **before inference**. Successful output means the runtime contract is visible and supported; it does not mean the checkpoint has passed integrity verification yet.


In [ ]:
import platform
from importlib.metadata import version

import torch

from moment_pipeline import (
    MomentConfig,
    PINNED_MODEL_ID,
    PINNED_REVISION,
    __version__ as moment_pipeline_version,
)

config = MomentConfig(task="reconstruction", device="cpu")
limits = config.limits
print("Python:", platform.python_version())
print("moment-pipeline:", moment_pipeline_version)
print("momentfm:", version("momentfm"))
print("PyTorch:", torch.__version__)
print("device:", config.resolved_device())
print("dtype:", config.dtype)
print("model:", PINNED_MODEL_ID)
print("immutable revision:", PINNED_REVISION)
print(
    "limits:",
    {
        "max_rows": limits.max_rows,
        "max_series": limits.max_series,
        "max_channels": limits.max_channels,
        "max_windows": limits.max_windows,
        "sequence_length": config.sequence_length,
        "patch_length": config.patch_length,
    },
)


## 3. Load the deterministic clean sample or BYOD

The default sample is generated locally and digest-verified. The optional upload path first validates the **raw CSV header** with `read_long_csv_bytes()` so duplicate or ambiguous names cannot be silently renamed by pandas; the resulting frame still goes through production validation in the next stage. For meaningful artificial-mask evaluation, the chosen first canonical window must contain at least one complete 8-step patch of source-observed values. Successful completion means one identified input frame is available and its source/digest are recorded.


In [ ]:
import hashlib
import json

import pandas as pd

from moment_pipeline import read_long_csv_bytes

USE_BYOD = False  # @param {type:"boolean"}
if USE_BYOD:
    try:
        from google.colab import files
    except ImportError as exc:
        raise RuntimeError(
            "BYOD upload is available in Colab. Outside Colab, load a CSV into `frame` "
            "with columns series_id,timestamp,channel,value and validate it before inference."
        ) from exc
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one CSV with columns series_id,timestamp,channel,value.")
    name, payload = next(iter(uploaded.items()))
    frame = read_long_csv_bytes(payload)
    sample_identity = {
        "kind": "byod",
        "name": name,
        "sha256": hashlib.sha256(payload).hexdigest(),
    }
    print(f"Loaded BYOD: {name}")
else:
    sample_root = ROOT / "examples" / "sample-data"
    subprocess.run([sys.executable, str(sample_root / "generate_samples.py")], check=True)
    sample_path = sample_root / "moment_clean.csv"
    manifest = {}
    for line in (sample_root / "SHA256SUMS").read_text(encoding="utf-8").splitlines():
        digest, filename = line.split("  ", 1)
        manifest[filename] = digest
    observed = hashlib.sha256(sample_path.read_bytes()).hexdigest()
    assert observed == manifest[sample_path.name], "sample digest mismatch"
    frame = pd.read_csv(sample_path)
    sample_identity = {"kind": "synthetic", "name": sample_path.name, "sha256": observed}
    print(f"Loaded verified synthetic sample: {sample_path}")
print(frame.head())
print("sample identity:", sample_identity)


## 4. Validate, canonicalize, and create a known-truth patch holdout

The repository validates the long-format schema and canonicalizes each series. The tutorial then selects the latest complete 8-step patch in the first window for which every channel has source-observed truth. This separates source-missing data from artificial evaluation masking. Successful output means validation passed, all canonicalization effects are disclosed, and the artificial holdout consists only of genuine known truth.


In [ ]:
import numpy as np

from moment_pipeline import to_windows, validate_long_frame

report, normalized = validate_long_frame(frame, config)
windows = to_windows(normalized, config, report=report, frame=normalized)
print(
    "validation:",
    {
        "rows": report.n_rows,
        "series": len(report.series_ids),
        "channels": len(report.channels),
        "irregular_series": list(report.irregular_series),
    },
)
print("window tensor:", windows.x_enc.shape)
print("padded windows:", int(sum(windows.padded)), "/", windows.n_windows)
print("truncated windows:", int(sum(windows.truncated)), "/", windows.n_windows)
if any(windows.truncated):
    print("WARNING: long input series were truncated to their final 512 timestamps.")
if any(windows.padded):
    print("NOTE: short input series were left-padded; padding is excluded from evaluation.")

visible = np.ones_like(windows.input_mask, dtype=np.float32)
complete_patch_starts = []
for candidate in range(0, windows.sequence_length, windows.patch_length):
    stop = candidate + windows.patch_length
    if np.all(windows.input_mask[0, candidate:stop] == 1) and np.all(
        windows.point_mask[0, :, candidate:stop] == 1
    ):
        complete_patch_starts.append(candidate)
if not complete_patch_starts:
    raise ValueError(
        "The first canonical window has no complete 8-step source-observed patch to hold out. "
        "Provide a series with at least 8 consecutive observed timestamps."
    )
start = complete_patch_starts[-1]
stop = start + windows.patch_length
visible[0, start:stop] = 0.0
print("artificial holdout window:", windows.window_ids[0])
print("deliberately hidden positions:", start, "through", stop - 1)
print("held-out point count across channels:", windows.n_channels * windows.patch_length)


## 5. Run pinned reconstruction and evaluate withheld truth

`load_moment()` integrity-verifies the immutable checkpoint. `impute()` receives the explicit visibility mask, and `masked_point_metrics()` computes MAE and RMSE **only** on deliberately hidden cells that had genuine source truth. **MAE** is average absolute error in the original units; **RMSE** is also in the original units but weights larger errors more strongly. Successful output means the verified reconstruction path executed and the reported sample metrics use only the declared holdout; they are not stable estimates of domain performance.


In [ ]:
from moment_pipeline import build_provenance, impute, load_moment, masked_point_metrics

model = load_moment(task="reconstruction", device="cpu")
result = impute(windows, model, mask=visible, warmup=False)
metrics = masked_point_metrics(result)
provenance = build_provenance(model, windows, result)
print("effective model:", model.identity.name)
print("effective revision:", model.identity.revision)
print("verified weight file:", model.identity.weight_file_loaded)
print("tutorial masked-point MAE:", metrics.mae)
print("tutorial masked-point RMSE:", metrics.rmse)
print("scored held-out cells:", metrics.n)
print("source masked_point_fraction:", result.masked_point_fraction)
print("effective model_masked_point_fraction:", result.model_masked_point_fraction)
print("effective masked_patch_fraction:", result.masked_patch_fraction)


## 6. Compare a simple interpolation baseline

A linear interpolation baseline is evaluated on the **same artificially withheld positions**. It uses the source series with the held-out patch removed. Successful output provides a same-support descriptive comparison; the baseline is not used to tune or select MOMENT, and one sample comparison does not establish superiority.


In [ ]:
baseline_rows = []
for channel in windows.channels:
    source = normalized[
        (normalized["series_id"] == windows.series_ids[0])
        & (normalized["channel"] == channel)
    ].sort_values("timestamp").copy()
    held_timestamps = set(pd.Series(windows.timestamps[0][start:stop]).dropna().tolist())
    if not held_timestamps:
        continue
    source["is_holdout"] = source["timestamp"].isin(held_timestamps)
    truth = source.loc[source["is_holdout"], ["timestamp", "value"]].copy()
    baseline_input = source["value"].mask(source["is_holdout"])
    baseline_pred = baseline_input.interpolate(method="linear", limit_direction="both")
    pred = baseline_pred[source["is_holdout"]].to_numpy(dtype=float)
    for (_, truth_row), prediction in zip(truth.iterrows(), pred, strict=True):
        baseline_rows.append(
            {
                "channel": channel,
                "timestamp": truth_row["timestamp"],
                "truth": float(truth_row["value"]),
                "prediction": float(prediction),
            }
        )
baseline_frame = pd.DataFrame(baseline_rows)
if baseline_frame.empty or not np.isfinite(baseline_frame["prediction"]).all():
    print("baseline unavailable: held-out region could not be interpolated from neighboring data")
    baseline_metrics = None
else:
    errors = (
        baseline_frame["prediction"].to_numpy(dtype=float)
        - baseline_frame["truth"].to_numpy(dtype=float)
    )
    baseline_metrics = {
        "mae": float(np.mean(np.abs(errors))),
        "rmse": float(np.sqrt(np.mean(errors**2))),
    }
    print("linear-interpolation tutorial baseline:", baseline_metrics)


## 7. Visualize original versus imputed values

This diagnostic plot compares the source series with the returned imputed product. `imputed_value` replaces only source-missing or deliberately hidden cells; observed neighbors hidden from the model only because they share the patch are not overwritten. Successful output makes that product behavior visually inspectable but does not replace the machine-readable export.


In [ ]:
def write_line_svg(path, layers, *, title, width=760, height=280):
    all_values = [float(value) for _, values in layers for value in values]
    low, high = min(all_values), max(all_values)
    span = high - low or 1.0
    max_points = max(len(values) for _, values in layers)
    left, right, top, bottom = 48, width - 20, 30, height - 38

    def point(index, value):
        x = left + (right - left) * index / max(max_points - 1, 1)
        y = bottom - (bottom - top) * (float(value) - low) / span
        return f"{x:.1f},{y:.1f}"

    strokes = ["#111827", "#2563eb", "#dc2626"]
    svg = [
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}">',
        f'<text x="{left}" y="18" font-family="sans-serif" font-size="14">{title}</text>',
    ]
    for idx, (label, values) in enumerate(layers):
        points = " ".join(point(i, value) for i, value in enumerate(values))
        stroke = strokes[idx % len(strokes)]
        svg.append(
            f'<polyline fill="none" stroke="{stroke}" stroke-width="2" points="{points}"/>'
        )
        svg.append(
            f'<text x="{left + 180 * idx}" y="{height - 10}" font-family="sans-serif" '
            f'font-size="12" fill="{stroke}">{label}</text>'
        )
    svg.append("</svg>")
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text("\n".join(svg), encoding="utf-8")
    return path

imputed_frame = result.to_frame()
channel = windows.channels[0]
view = imputed_frame[imputed_frame["channel"] == channel].tail(96)
plot_path = write_line_svg(
    ROOT / "outputs" / "moment_imputation.svg",
    [
        ("original", view["original_value"].fillna(view["imputed_value"]).tolist()),
        ("imputed", view["imputed_value"].tolist()),
    ],
    title=f"MOMENT imputation — {channel}",
)
try:
    from IPython.display import SVG, display

    display(SVG(filename=str(plot_path)))
except ImportError:
    print(f"SVG written to {plot_path}")
print(
    imputed_frame.loc[
        imputed_frame["requested_hidden"],
        [
            "series_id",
            "timestamp",
            "channel",
            "original_value",
            "imputed_value",
            "model_hidden",
        ],
    ]
)


## 8. Export imputed series, metrics, baseline, and provenance

This stage writes the stable machine-readable handoff files: `outputs/moment_imputed_series.csv`, `outputs/moment_imputation_metrics.json`, and `outputs/moment_imputation_provenance.json` (plus the diagnostic SVG from the previous stage). Successful completion establishes that the imputed product, sample-evaluation procedure/baseline, and model/runtime/data provenance were serialized under documented filenames; file creation does **not** establish generalized model quality.


In [ ]:
output_dir = ROOT / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)
imputed_frame.to_csv(output_dir / "moment_imputed_series.csv", index=False)
metric_payload = {
    "estimation_procedure": "single deterministic artificial 8-step patch holdout",
    "sample_evidence_only": True,
    "n": metrics.n,
    "mae": metrics.mae,
    "rmse": metrics.rmse,
    "linear_interpolation_baseline": baseline_metrics,
}
(output_dir / "moment_imputation_metrics.json").write_text(
    json.dumps(metric_payload, indent=2),
    encoding="utf-8",
)
provenance["data"] = sample_identity
provenance["evaluation"] = metric_payload
(output_dir / "moment_imputation_provenance.json").write_text(
    json.dumps(provenance, indent=2, default=str),
    encoding="utf-8",
)
print("exports:", sorted(path.name for path in output_dir.glob("moment_imput*")))


## Interpretation, limits, and next steps

A successful run proves that the repository can validate this input, reconstruct with the pinned MOMENT checkpoint, keep source missingness separate from artificial evaluation masking, score only known withheld truth, preserve observed values in the imputed product, and export results and provenance.

It **does not prove** that the displayed MAE/RMSE generalize to other series or domains, that MOMENT beats the interpolation baseline reliably, or that the model supplies calibrated per-prediction uncertainty. This path returns point reconstructions only; no uncertainty interval is provided.

For stronger evidence, repeat domain-appropriate masking over an independent dataset, report dispersion across windows/series, and compare multiple baselines without using the evaluation set for model selection.
